# Titanic Survival Prediction using Machine Learning

## 1. Project Title
**Titanic Survival Prediction using Machine Learning**

## 2. Objective (Business Understanding)

The goal is to predict whether a passenger survived or not during the Titanic disaster. This is a binary classification problem with target variable `Survived` (0 = No, 1 = Yes).

Why prediction matters:
- It supports historical analysis and risk profiling.
- It helps understand which passenger features impacted survival.

How data helps decision-making:
- Data reveals patterns that are not obvious by intuition alone.
- A trained model converts these patterns into predictions on new data.

## 3. Problem Statement

Given passenger details such as Age, Sex, Pclass, Fare, etc., predict survival.

Target variable: `Survived` (0 = No, 1 = Yes)

## 4. Dataset Understanding

Data description:
- Total number of rows (passengers): `len(df)`
- Total number of columns (features): `df.shape[1]`

Features included:
- Pclass → Passenger class
- Sex → Gender
- Age → Age
- Fare → Ticket price
- Embarked → Boarding port
- SibSp → Siblings/Spouse aboard
- Parch → Parents/Children aboard
- Survived → Target

Each row = one passenger
Each column = one attribute

## 5. Data Exploration (EDA)

Analyze:
- Survival distribution
- Gender vs survival
- Class vs survival
- Age distribution

Observations (examples):
- Females had higher survival rate.
- 1st class passengers survived more.

## 6. Data Cleaning & Preparation

Missing values:
- Identify with `df.isna().sum()`
- Handle Age with median imputation
- Handle Embarked with mode imputation

Dropped irrelevant columns:
- PassengerId, Name, Ticket, Cabin
- Justification: these do not contribute directly to the survival prediction model.

## 7. Feature Engineering

Convert categorical features to numeric:
- Sex (female=0, male=1) or one-hot
- Embarked (C/Q/S) as one-hot encoded or label encoded

Explanation: Machine learning models require numeric input.

## 8. Feature vs Label Explanation

Features (Independent Variables): Pclass, Sex, Age, Fare, Embarked, SibSp, Parch.
Label (Dependent Variable): Survived.

The model learns mapping from features to survival outcome, predicting 0/1 for new passengers.

## 9. Train-Test Split

Split data:
- Training set: 80%
- Testing set: 20%

Explanation:
- Training: model learns patterns from known labels.
- Testing: evaluate how well the model generalizes.

## 10. Model Selection

Chosen model: Logistic Regression.

Why this model?
- Binary classification problem.
- Simple, interpretable output.
- Strong baseline for structured tabular data.

Advantages:
- Fast training and inference.
- Coefficients directly show feature impact.

## 11. Model Training

Train logistic regression on the training set. The model learns the relationship between passenger features and survival.

## 12. Model Evaluation

Use accuracy: `(TP + TN) / total`.
Interpretation: Accuracy of X% indicates the overall performance on the test set.

## 13. Results & Insights

Key findings:
- Gender effect: Women had higher survival chances.
- Class effect: 1st class passengers survived more.
- Age trends: Children and young adults tended to survive at higher rates.

Example summary:
- Women and children had higher survival chances.
- Higher class → higher survival.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from pathlib import Path

data_path = None
candidates = list(Path('.').rglob('*titanic*.csv'))
if candidates:
    data_path = candidates[0]
    print(f'Using dataset from: {data_path}')
    df = pd.read_csv(data_path)
else:
    print('No titanic CSV found; generating synthetic dataset with 100 rows for testing.')
    import numpy as np
    np.random.seed(42)
    n = 100
    df = pd.DataFrame({
        'PassengerId': np.arange(1, n + 1),
        'Survived': np.random.choice([0, 1], size=n, p=[0.62, 0.38]),
        'Pclass': np.random.choice([1, 2, 3], size=n, p=[0.24, 0.20, 0.56]),
        'Name': [f'Passenger {i}' for i in range(1, n + 1)],
        'Sex': np.random.choice(['male', 'female'], size=n, p=[0.64, 0.36]),
        'Age': np.random.choice(list(range(1, 81)) + [None], size=n),
        'SibSp': np.random.randint(0, 3, size=n),
        'Parch': np.random.randint(0, 3, size=n),
        'Ticket': [f'TKT{1000 + i}' for i in range(n)],
        'Fare': np.round(np.random.uniform(10, 250, size=n), 2),
        'Cabin': [None] * n,
        'Embarked': np.random.choice(['C', 'Q', 'S'], size=n, p=[0.2, 0.1, 0.7])
    })

# 4. Understanding
print('Rows:', len(df), 'Cols:', df.shape[1])
print(df.head())

# 5. EDA
print(df['Survived'].value_counts())
print(df.groupby('Sex')['Survived'].mean())
print(df.groupby('Pclass')['Survived'].mean())

# 6. Clean
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop(['PassengerId','Name','Ticket','Cabin'], axis=1, inplace=True)

# 7. Feature engineering
df['Sex'] = df['Sex'].map({'female':0, 'male':1})
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

X = df.drop('Survived', axis=1)
y = df['Survived']

# 9. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 10 & 11. Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 12. Evaluate
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')

# 13. Insights
coef_df = pd.DataFrame({'feature': X.columns, 'coef': model.coef_[0]})
print(coef_df.sort_values(by='coef', ascending=False))

No titanic CSV found; generating synthetic dataset with 100 rows for testing.
Rows: 100 Cols: 12
   PassengerId  Survived  Pclass         Name     Sex Age  SibSp  Parch  \
0            1         0       1  Passenger 1  female  47      2      2   
1            2         1       3  Passenger 2    male  55      0      1   
2            3         1       2  Passenger 3    male  40      2      2   
3            4         0       3  Passenger 4  female  52      1      0   
4            5         0       3  Passenger 5    male  16      0      0   

    Ticket    Fare Cabin Embarked  
0  TKT1000  209.00  None        S  
1  TKT1001  146.58  None        Q  
2  TKT1002   25.24  None        S  
3  TKT1003   18.84  None        S  
4  TKT1004   42.12  None        S  
Survived
0    66
1    34
Name: count, dtype: int64
Sex
female    0.317073
male      0.355932
Name: Survived, dtype: float64
Pclass
1    0.375000
2    0.391304
3    0.301887
Name: Survived, dtype: float64
Accuracy: 0.5500
      feature  

C:\Users\sai\AppData\Local\Temp\ipykernel_16232\1204316356.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\sai\AppData\Local\Temp\ipykernel_16232\1204316356.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Age'].fillna(df['Age'].median(), 

## Dataset source

If you need the real Titanic dataset, download from Kaggle:
https://www.kaggle.com/c/titanic/data

You can put the file into this folder and rerun the notebook.

In [2]:
# Cell 1: Imports + dataset loading (real or synthetic)
import pandas as pd
from pathlib import Path

kaggle_link = 'https://www.kaggle.com/c/titanic/data'

candidates = list(Path('.').rglob('*titanic*.csv'))
if candidates:
    data_path = candidates[0]
    print(f'Using dataset from: {data_path}')
    df = pd.read_csv(data_path)
else:
    print('No titanic CSV found; generating synthetic dataset with 100 rows for testing.')
    import numpy as np
    np.random.seed(42)
    n = 100
    df = pd.DataFrame({
        'PassengerId': np.arange(1, n + 1),
        'Survived': np.random.choice([0, 1], size=n, p=[0.62, 0.38]),
        'Pclass': np.random.choice([1, 2, 3], size=n, p=[0.24, 0.20, 0.56]),
        'Name': [f'Passenger {i}' for i in range(1, n + 1)],
        'Sex': np.random.choice(['male', 'female'], size=n, p=[0.64, 0.36]),
        'Age': np.random.choice(list(range(1, 81)) + [None], size=n),
        'SibSp': np.random.randint(0, 3, size=n),
        'Parch': np.random.randint(0, 3, size=n),
        'Ticket': [f'TKT{1000 + i}' for i in range(n)],
        'Fare': np.round(np.random.uniform(10, 250, size=n), 2),
        'Cabin': [None] * n,
        'Embarked': np.random.choice(['C', 'Q', 'S'], size=n, p=[0.2, 0.1, 0.7])
    })

print('Rows:', len(df), 'Cols:', df.shape[1])
print(df.head())

No titanic CSV found; generating synthetic dataset with 100 rows for testing.
Rows: 100 Cols: 12
   PassengerId  Survived  Pclass         Name     Sex Age  SibSp  Parch  \
0            1         0       1  Passenger 1  female  47      2      2   
1            2         1       3  Passenger 2    male  55      0      1   
2            3         1       2  Passenger 3    male  40      2      2   
3            4         0       3  Passenger 4  female  52      1      0   
4            5         0       3  Passenger 5    male  16      0      0   

    Ticket    Fare Cabin Embarked  
0  TKT1000  209.00  None        S  
1  TKT1001  146.58  None        Q  
2  TKT1002   25.24  None        S  
3  TKT1003   18.84  None        S  
4  TKT1004   42.12  None        S  


In [ ]:
# Cell 2: Data cleaning + preprocessing
from sklearn.model_selection import train_test_split

# 6. Clean
if 'Age' in df.columns:
    df['Age'].fillna(df['Age'].median(), inplace=True)
if 'Embarked' in df.columns:
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

if set(['PassengerId', 'Name', 'Ticket', 'Cabin']).issubset(df.columns):
    df.drop(['PassengerId','Name','Ticket','Cabin'], axis=1, inplace=True)

# 7. Feature engineering
if 'Sex' in df.columns:
    df['Sex'] = df['Sex'].map({'female':0, 'male':1})
if 'Embarked' in df.columns:
    df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

# Explicit catch-all imputation in case any column still has NaN
from sklearn.impute import SimpleImputer
num_cols = df.select_dtypes(include=['number']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
if num_cols:
    num_imputer = SimpleImputer(strategy='median')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])
if cat_cols:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

X = df.drop('Survived', axis=1)
y = df['Survived']

print('Missing values after imputation:')
print(df.isna().sum())

print('Cleaned features:', X.columns.tolist())
print('Sample rows after encoding:')
print(X.head())

Cleaned features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q', 'Embarked_S']
Sample rows after encoding:
   Pclass  Sex   Age  SibSp  Parch    Fare  Embarked_Q  Embarked_S
0       1    0  47.0      2      2  209.00       False        True
1       3    1  55.0      0      1  146.58        True       False
2       2    1  40.0      2      2   25.24       False        True
3       3    0  52.0      1      0   18.84       False        True
4       3    1  16.0      0      0   42.12       False        True


C:\Users\sai\AppData\Local\Temp\ipykernel_16232\2929467524.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\sai\AppData\Local\Temp\ipykernel_16232\2929467524.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Age'].fillna(df['Age'].median(), in

In [6]:
# Cell 3: Train-test split + model training
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# sanity check that there are no missing values in training set
assert not X_train.isna().any().any(), "X_train still has NaN; check imputation pipeline"
assert not y_train.isna().any(), "y_train still has NaN"

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print('Training complete')

Training complete


In [5]:
# Cell 4: Evaluation + insight reporting
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')

coef_df = pd.DataFrame({'feature': X.columns, 'coef': model.coef_[0]})
print('Top positive impact features:')
print(coef_df.sort_values(by='coef', ascending=False).head())
print('Top negative impact features:')
print(coef_df.sort_values(by='coef', ascending=True).head())

Accuracy: 0.5500
Top positive impact features:
  feature      coef
1     Sex  0.656996
3   SibSp  0.576946
4   Parch  0.541512
2     Age  0.018292
5    Fare  0.000064
Top negative impact features:
      feature      coef
7  Embarked_S -0.972214
6  Embarked_Q -0.282074
0      Pclass -0.245513
5        Fare  0.000064
2         Age  0.018292
